# Notebook 04: From REINFORCE to PPO

**Series**: Frontier ML Interview Prep -- RLHF Deep Dive

**Goal**: Build the full policy gradient story: REINFORCE -> variance reduction with baselines -> Actor-Critic -> GAE -> PPO. All implemented from scratch on CartPole.

---

## 1. Self-Quiz (Active Recall)

Before reading any material, answer these from memory. Write your answers in the empty cell below.

1. **State the policy gradient theorem.** Write the gradient of the expected return.
2. **What is the main problem with REINFORCE?** Why is it impractical for complex tasks?
3. **What does a baseline do?** Why does subtracting a baseline not introduce bias?
4. **What is the advantage function $A(s,a)$?** Write its definition.
5. **What does PPO's clipping do?** Write the clipped surrogate objective.
6. **What is importance sampling** and why does PPO need it?
7. **What is GAE?** What are the two parameters $\gamma$ and $\lambda$?

*Your answers here (double-click to edit):*

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...
7. ...

---
## 2. Setup

In [ ]:
# Install dependencies (Colab-compatible)
!pip install -q torch gymnasium matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 3. Policy Gradient Intuition

### The Key Idea

We have a parameterized policy $\pi_\theta(a|s)$ that maps states to action distributions. Our objective is to maximize expected cumulative reward:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]$$

The **policy gradient theorem** (Sutton et al., 1999) tells us how to differentiate this:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

where $G_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$ is the **return from time $t$**.

### Intuition: The Log-Derivative Trick

- $\nabla_\theta \log \pi_\theta(a|s)$ points in the direction that **increases the probability of action $a$ in state $s$**
- $G_t$ is the **reward signal**: positive means "do more of this", negative means "do less"
- The product: increase probability of actions that led to high reward, decrease for low reward

This is beautifully simple: **trial and error, formalized as gradient ascent.**

In [ ]:
# Visualize the policy gradient idea with a simple 2-action example

# Imagine state s with two actions: left and right
# Policy: pi(left|s) = sigmoid(theta), pi(right|s) = 1 - sigmoid(theta)

thetas = np.linspace(-4, 4, 200)
pi_left = 1 / (1 + np.exp(-thetas))  # sigmoid
pi_right = 1 - pi_left

# Suppose: taking 'left' gives reward +1, 'right' gives reward -1
# Expected return: J(theta) = pi(left) * 1 + pi(right) * (-1) = 2*sigmoid(theta) - 1
J = 2 * pi_left - 1

# Gradient: dJ/dtheta = 2 * sigmoid(theta) * (1-sigmoid(theta))
dJ = 2 * pi_left * (1 - pi_left)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thetas, pi_left, 'b-', linewidth=2, label='pi(left|s)')
ax1.plot(thetas, pi_right, 'r-', linewidth=2, label='pi(right|s)')
ax1.set_xlabel('theta', fontsize=12)
ax1.set_ylabel('Probability', fontsize=12)
ax1.set_title('Policy probabilities', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(thetas, J, 'g-', linewidth=2, label='J(theta) = E[R]')
ax2.plot(thetas, dJ, 'm--', linewidth=2, label='dJ/dtheta (gradient)')
ax2.set_xlabel('theta', fontsize=12)
ax2.set_title('Expected return & its gradient', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: the gradient is always positive -- it always pushes theta")
print("toward making pi(left) larger, because left gives higher reward.")
print("Policy gradient = trial and error, formalized as gradient ascent.")

---
## 4. REINFORCE on CartPole

REINFORCE (Williams, 1992) is the simplest policy gradient algorithm:
1. Collect a full trajectory using current policy
2. Compute returns $G_t$ for each timestep
3. Update: $\theta \leftarrow \theta + \alpha \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) G_t$

In [ ]:
class PolicyNetwork(nn.Module):
    """Simple 2-layer MLP policy for discrete action spaces."""
    
    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )
    
    def forward(self, obs: torch.Tensor) -> Categorical:
        """Return action distribution given observation."""
        logits = self.net(obs)
        return Categorical(logits=logits)
    
    def get_action(self, obs: np.ndarray):
        """Sample action and return (action, log_prob)."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(next(self.parameters()).device)
        dist = self.forward(obs_t)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob


# Verify on CartPole
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]  # 4
act_dim = env.action_space.n  # 2
print(f"CartPole: obs_dim={obs_dim}, act_dim={act_dim}")

policy = PolicyNetwork(obs_dim, act_dim).to(device)
obs, _ = env.reset(seed=42)
action, log_prob = policy.get_action(obs)
print(f"Test action: {action}, log_prob: {log_prob.item():.4f}")

In [ ]:
class REINFORCE:
    """Vanilla REINFORCE (Monte Carlo Policy Gradient).
    
    The simplest policy gradient method:
    1. Collect full episode
    2. Compute discounted returns G_t
    3. Policy gradient: nabla J = E[sum_t nabla log pi(a_t|s_t) * G_t]
    """
    
    def __init__(self, obs_dim, act_dim, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.policy = PolicyNetwork(obs_dim, act_dim).to(device)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)
    
    def compute_returns(self, rewards):
        """Compute discounted returns G_t = sum_{t'>=t} gamma^(t'-t) * r_{t'}."""
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        # Normalize returns (helps with training stability)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        return returns
    
    def collect_episode(self, env):
        """Run one episode and collect (log_probs, rewards)."""
        log_probs = []
        rewards = []
        
        obs, _ = env.reset()
        done = False
        
        while not done:
            action, log_prob = self.policy.get_action(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            log_probs.append(log_prob)
            rewards.append(reward)
        
        return log_probs, rewards
    
    def update(self, log_probs, rewards):
        """REINFORCE update: loss = -sum(log_pi * G_t)."""
        returns = self.compute_returns(rewards)
        
        # Policy gradient loss (negative because we minimize in PyTorch)
        policy_loss = 0
        for log_prob, G in zip(log_probs, returns):
            policy_loss += -log_prob * G  # gradient ascent via minimizing negative
        
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        
        return policy_loss.item()


def train_agent(agent, env_name='CartPole-v1', num_episodes=1000, print_every=100):
    """Train any policy gradient agent and return episode rewards."""
    env = gym.make(env_name)
    episode_rewards = []
    running_reward = 0
    
    for ep in range(num_episodes):
        log_probs, rewards = agent.collect_episode(env)
        loss = agent.update(log_probs, rewards)
        
        ep_reward = sum(rewards)
        episode_rewards.append(ep_reward)
        running_reward = 0.95 * running_reward + 0.05 * ep_reward
        
        if (ep + 1) % print_every == 0:
            print(f"Episode {ep+1:4d} | Reward: {ep_reward:6.1f} | Running: {running_reward:6.1f}")
        
        # Solved!
        if running_reward >= 475:
            print(f"\nSolved at episode {ep+1}! Running reward: {running_reward:.1f}")
            break
    
    env.close()
    return episode_rewards

In [ ]:
# Train REINFORCE on CartPole
print("=" * 60)
print("Training REINFORCE on CartPole-v1")
print("=" * 60)

reinforce_agent = REINFORCE(obs_dim=4, act_dim=2, lr=1e-3, gamma=0.99)
reinforce_rewards = train_agent(reinforce_agent, num_episodes=1000, print_every=100)

print(f"\nFinal running average: {np.mean(reinforce_rewards[-100:]):.1f}")
print(f"\nProblem: notice the HIGH VARIANCE in episode rewards!")
print(f"Std of last 100 episodes: {np.std(reinforce_rewards[-100:]):.1f}")

In [ ]:
# Plot REINFORCE training curve

def plot_rewards(rewards_dict, title="Training Curves", window=50):
    """Plot episode rewards with smoothed curves for multiple agents."""
    fig, ax = plt.subplots(figsize=(12, 5))
    
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']
    
    for i, (name, rewards) in enumerate(rewards_dict.items()):
        color = colors[i % len(colors)]
        ax.plot(rewards, alpha=0.15, color=color)
        # Smoothed
        if len(rewards) >= window:
            smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
            ax.plot(range(window-1, len(rewards)), smoothed, 
                    color=color, linewidth=2.5, label=f'{name} (smoothed)')
    
    ax.axhline(y=475, color='gray', linestyle='--', alpha=0.5, label='Solved threshold')
    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Episode Reward', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_rewards({'REINFORCE': reinforce_rewards}, 'REINFORCE on CartPole-v1')

### The Problem with REINFORCE: High Variance

Look at the raw reward curve -- it's extremely noisy. This is because:

1. **Monte Carlo returns** $G_t$ are estimated from a single trajectory -- high variance
2. **Credit assignment**: Every action gets credit proportional to the *entire future return*, even though most of the return may be due to later actions
3. **Sample inefficiency**: We need many episodes to get a reliable gradient estimate

Solution: **subtract a baseline** to reduce variance without introducing bias.

---
## 5. Adding a Baseline (Variance Reduction)

### The Key Theorem

We can subtract any function $b(s)$ that depends only on the state from the return without changing the expected gradient:

$$\nabla_\theta J = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))\right]$$

**Why is this still unbiased?**

$$\mathbb{E}_{a \sim \pi}[\nabla_\theta \log \pi_\theta(a|s) \cdot b(s)] = b(s) \sum_a \nabla_\theta \pi_\theta(a|s) = b(s) \cdot \nabla_\theta \underbrace{\sum_a \pi_\theta(a|s)}_{=1} = 0$$

The gradient of a constant (1) is 0. So the baseline vanishes in expectation.

**Best baseline**: The optimal baseline (minimizing variance) is close to $V^\pi(s)$. This gives us the **advantage function**:

$$A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s) \approx G_t - V^\pi(s_t)$$

Intuition: "How much better was this action compared to the average action in this state?"

In [ ]:
class ValueNetwork(nn.Module):
    """State value function V(s) -- used as baseline for variance reduction."""
    
    def __init__(self, obs_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs).squeeze(-1)


class REINFORCEWithBaseline:
    """REINFORCE with a learned value function baseline.
    
    Advantage A_t = G_t - V(s_t) reduces variance dramatically.
    """
    
    def __init__(self, obs_dim, act_dim, lr_policy=1e-3, lr_value=1e-3, gamma=0.99):
        self.gamma = gamma
        self.policy = PolicyNetwork(obs_dim, act_dim).to(device)
        self.value_fn = ValueNetwork(obs_dim).to(device)
        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr_policy)
        self.value_optimizer = torch.optim.Adam(self.value_fn.parameters(), lr=lr_value)
    
    def compute_returns(self, rewards):
        """Compute discounted returns (no normalization -- baseline handles it)."""
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        return torch.FloatTensor(returns).to(device)
    
    def collect_episode(self, env):
        """Run one episode and collect transitions (log_probs, rewards, states)."""
        log_probs = []
        rewards = []
        states = []
        
        obs, _ = env.reset()
        done = False
        
        while not done:
            # BUG FIX: Use obs.copy() to avoid stale references if env
            # modifies obs in-place on subsequent step() calls.
            states.append(obs.copy())
            action, log_prob = self.policy.get_action(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            log_probs.append(log_prob)
            rewards.append(reward)
        
        return log_probs, rewards, states
    
    def update(self, log_probs, rewards, states=None):
        """Update policy using advantage = return - baseline."""
        returns = self.compute_returns(rewards)
        states_t = torch.FloatTensor(np.array(states)).to(device)
        
        # Compute baseline values
        values = self.value_fn(states_t)
        
        # Advantage = return - baseline (detach baseline from policy gradient)
        advantages = returns - values.detach()
        # Normalize advantages
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Policy loss: -log_pi * advantage
        policy_loss = 0
        for log_prob, adv in zip(log_probs, advantages):
            policy_loss += -log_prob * adv
        
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()
        
        # Value loss: MSE(V(s), G_t)
        value_loss = F.mse_loss(values, returns)
        
        self.value_optimizer.zero_grad()
        value_loss.backward()
        self.value_optimizer.step()
        
        return policy_loss.item()


def train_agent_with_states(agent, env_name='CartPole-v1', num_episodes=1000, print_every=100):
    """Train agent that also collects states (for baseline methods)."""
    env = gym.make(env_name)
    episode_rewards = []
    running_reward = 0
    
    for ep in range(num_episodes):
        log_probs, rewards, states = agent.collect_episode(env)
        loss = agent.update(log_probs, rewards, states)
        
        ep_reward = sum(rewards)
        episode_rewards.append(ep_reward)
        running_reward = 0.95 * running_reward + 0.05 * ep_reward
        
        if (ep + 1) % print_every == 0:
            print(f"Episode {ep+1:4d} | Reward: {ep_reward:6.1f} | Running: {running_reward:6.1f}")
        
        if running_reward >= 475:
            print(f"\nSolved at episode {ep+1}! Running reward: {running_reward:.1f}")
            break
    
    env.close()
    return episode_rewards

In [ ]:
# Train REINFORCE with baseline
print("=" * 60)
print("Training REINFORCE + Baseline on CartPole-v1")
print("=" * 60)

baseline_agent = REINFORCEWithBaseline(obs_dim=4, act_dim=2, lr_policy=1e-3, lr_value=1e-2, gamma=0.99)
baseline_rewards = train_agent_with_states(baseline_agent, num_episodes=1000, print_every=100)

print(f"\nVariance comparison (last 100 episodes):")
print(f"  REINFORCE:          std = {np.std(reinforce_rewards[-100:]):.1f}")
print(f"  REINFORCE+Baseline: std = {np.std(baseline_rewards[-100:]):.1f}")

In [ ]:
# Compare REINFORCE vs REINFORCE + Baseline
plot_rewards(
    {'REINFORCE': reinforce_rewards, 'REINFORCE + Baseline': baseline_rewards},
    'Variance Reduction with Baseline'
)

---
## 6. Actor-Critic with GAE

### From REINFORCE to Actor-Critic

REINFORCE (even with baseline) still has a problem: it uses **Monte Carlo returns** $G_t$, which require waiting until the end of the episode. Actor-Critic methods use **bootstrapped estimates** via the value function.

### Generalized Advantage Estimation (GAE)

GAE (Schulman et al., 2016) provides a smooth interpolation between:
- **TD(0)**: $\hat{A}_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ -- low variance, high bias
- **MC**: $\hat{A}_t = G_t - V(s_t)$ -- high variance, low bias

The GAE estimator:

$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{T-t} (\gamma\lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the **TD residual**.

- $\lambda = 0$: Pure TD (low variance, high bias)
- $\lambda = 1$: Pure MC (high variance, low bias)
- $\lambda \in (0,1)$: Smooth trade-off. Typical: $\lambda = 0.95$

In [ ]:
class ActorCritic:
    """Actor-Critic with Generalized Advantage Estimation (GAE).
    
    Key improvements over REINFORCE+Baseline:
    - Uses TD residuals instead of MC returns for advantage estimation
    - GAE provides bias-variance trade-off via lambda
    - Can learn from partial trajectories (online)
    """
    
    def __init__(self, obs_dim, act_dim, lr_actor=3e-4, lr_critic=1e-3,
                 gamma=0.99, gae_lambda=0.95):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        
        self.policy = PolicyNetwork(obs_dim, act_dim).to(device)
        self.value_fn = ValueNetwork(obs_dim).to(device)
        
        self.actor_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr_actor)
        self.critic_optimizer = torch.optim.Adam(self.value_fn.parameters(), lr=lr_critic)
    
    def compute_gae(self, rewards, values, next_value, dones):
        """Compute Generalized Advantage Estimation.
        
        GAE(gamma, lambda) = sum_{l=0}^{T-t} (gamma*lambda)^l * delta_{t+l}
        where delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
        """
        advantages = []
        gae = 0
        
        # Process in reverse order
        values_extended = list(values) + [next_value]
        
        for t in reversed(range(len(rewards))):
            # TD residual: delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)
            if dones[t]:
                delta = rewards[t] - values_extended[t]
                gae = delta  # reset GAE at episode boundary
            else:
                delta = rewards[t] + self.gamma * values_extended[t + 1] - values_extended[t]
                gae = delta + self.gamma * self.gae_lambda * gae
            
            advantages.insert(0, gae)
        
        return torch.FloatTensor(advantages).to(device)
    
    def collect_episode(self, env):
        """Collect one episode of transitions."""
        states, actions, rewards, log_probs, dones = [], [], [], [], []
        
        obs, _ = env.reset()
        done = False
        
        while not done:
            states.append(obs.copy())
            action, log_prob = self.policy.get_action(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            # NOTE: treating truncation like termination zeroes the bootstrap at
            # time-limit cutoffs, which biases value targets; correct handling
            # bootstraps V(s_next) when truncated (zero only on true termination).
            done = terminated or truncated
            
            actions.append(action)
            rewards.append(reward)
            log_probs.append(log_prob)
            dones.append(done)
        
        return states, actions, rewards, log_probs, dones
    
    def update(self, log_probs, rewards, states=None, actions=None, dones=None):
        """Actor-Critic update with GAE advantages."""
        states_t = torch.FloatTensor(np.array(states)).to(device)
        
        # Get value predictions
        with torch.no_grad():
            values = self.value_fn(states_t).cpu().numpy()
        
        # Compute GAE advantages
        advantages = self.compute_gae(rewards, values, 0.0, dones)
        returns = advantages + torch.FloatTensor(values).to(device)
        
        # Normalize advantages
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Actor (policy) loss
        actor_loss = 0
        for log_prob, adv in zip(log_probs, advantages):
            actor_loss += -log_prob * adv.detach()
        actor_loss = actor_loss / len(log_probs)
        
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # Critic (value function) loss
        values_pred = self.value_fn(states_t)
        critic_loss = F.mse_loss(values_pred, returns.detach())
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        return actor_loss.item()


def train_actor_critic(agent, env_name='CartPole-v1', num_episodes=1000, print_every=100):
    """Train Actor-Critic agent."""
    env = gym.make(env_name)
    episode_rewards = []
    running_reward = 0
    
    for ep in range(num_episodes):
        states, actions, rewards, log_probs, dones = agent.collect_episode(env)
        loss = agent.update(log_probs, rewards, states=states, actions=actions, dones=dones)
        
        ep_reward = sum(rewards)
        episode_rewards.append(ep_reward)
        running_reward = 0.95 * running_reward + 0.05 * ep_reward
        
        if (ep + 1) % print_every == 0:
            print(f"Episode {ep+1:4d} | Reward: {ep_reward:6.1f} | Running: {running_reward:6.1f}")
        
        if running_reward >= 475:
            print(f"\nSolved at episode {ep+1}! Running reward: {running_reward:.1f}")
            break
    
    env.close()
    return episode_rewards

In [ ]:
# Train Actor-Critic with GAE
print("=" * 60)
print("Training Actor-Critic (GAE) on CartPole-v1")
print("=" * 60)

ac_agent = ActorCritic(obs_dim=4, act_dim=2, lr_actor=3e-4, lr_critic=1e-3,
                        gamma=0.99, gae_lambda=0.95)
ac_rewards = train_actor_critic(ac_agent, num_episodes=1000, print_every=100)

---
## 7. PPO: Clipped Surrogate Objective

### The Problem PPO Solves

With standard policy gradients, a **large update** can be catastrophic:
- The policy changes dramatically in one step
- The old data (collected under $\pi_{\text{old}}$) becomes invalid under $\pi_{\text{new}}$
- Performance can collapse and never recover

### Importance Sampling

To reuse data collected under $\pi_{\text{old}}$ for updating $\pi_{\text{new}}$, we use **importance sampling**:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

The surrogate objective becomes:

$$L^{\text{IS}}(\theta) = \mathbb{E}_t\left[r_t(\theta) \cdot \hat{A}_t\right]$$

### The PPO Clipped Objective

PPO clips the importance sampling ratio to prevent too-large updates:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

where $\epsilon = 0.2$ typically.

### How Clipping Works

| Scenario | Advantage | Ratio | Clipping effect |
|----------|-----------|-------|-----------------|
| Good action, ratio too high | $A > 0$ | $r > 1+\epsilon$ | Clips -- prevents making this action *too* probable |
| Good action, ratio normal | $A > 0$ | $1-\epsilon < r < 1+\epsilon$ | No clip -- normal update |
| Bad action, ratio too low | $A < 0$ | $r < 1-\epsilon$ | Clips -- prevents making this action *too* improbable |
| Bad action, ratio normal | $A < 0$ | $1-\epsilon < r < 1+\epsilon$ | No clip -- normal update |

In [ ]:
# Visualize the PPO clipping mechanism

ratios = np.linspace(0.0, 2.5, 300)
epsilon = 0.2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Case 1: Positive advantage (A > 0) -- good action, want to increase probability
A_pos = 1.0
unclipped_pos = ratios * A_pos
clipped_pos = np.clip(ratios, 1 - epsilon, 1 + epsilon) * A_pos
ppo_pos = np.minimum(unclipped_pos, clipped_pos)

ax1.plot(ratios, unclipped_pos, 'b--', linewidth=2, label='Unclipped: r * A', alpha=0.7)
ax1.plot(ratios, clipped_pos, 'r--', linewidth=2, label=f'Clipped: clip(r,{1-epsilon},{1+epsilon}) * A', alpha=0.7)
ax1.plot(ratios, ppo_pos, 'g-', linewidth=3, label='PPO: min(unclipped, clipped)')
ax1.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax1.axvline(x=1-epsilon, color='orange', linestyle='--', alpha=0.5, label=f'r={1-epsilon}')
ax1.axvline(x=1+epsilon, color='orange', linestyle='--', alpha=0.5, label=f'r={1+epsilon}')
ax1.fill_between(ratios, ppo_pos, alpha=0.1, color='green')
ax1.set_xlabel('Importance sampling ratio r(theta)', fontsize=11)
ax1.set_ylabel('Objective', fontsize=11)
ax1.set_title('Positive Advantage (A > 0)\n"Good action -- increase probability"', fontsize=12)
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.5, 3.0)

# Case 2: Negative advantage (A < 0) -- bad action, want to decrease probability
A_neg = -1.0
unclipped_neg = ratios * A_neg
clipped_neg = np.clip(ratios, 1 - epsilon, 1 + epsilon) * A_neg
ppo_neg = np.minimum(unclipped_neg, clipped_neg)

ax2.plot(ratios, unclipped_neg, 'b--', linewidth=2, label='Unclipped: r * A', alpha=0.7)
ax2.plot(ratios, clipped_neg, 'r--', linewidth=2, label=f'Clipped: clip(r,{1-epsilon},{1+epsilon}) * A', alpha=0.7)
ax2.plot(ratios, ppo_neg, 'g-', linewidth=3, label='PPO: min(unclipped, clipped)')
ax2.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax2.axvline(x=1-epsilon, color='orange', linestyle='--', alpha=0.5, label=f'r={1-epsilon}')
ax2.axvline(x=1+epsilon, color='orange', linestyle='--', alpha=0.5, label=f'r={1+epsilon}')
ax2.fill_between(ratios, ppo_neg, alpha=0.1, color='green')
ax2.set_xlabel('Importance sampling ratio r(theta)', fontsize=11)
ax2.set_ylabel('Objective', fontsize=11)
ax2.set_title('Negative Advantage (A < 0)\n"Bad action -- decrease probability"', fontsize=12)
ax2.legend(fontsize=9, loc='lower left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-3.0, 0.5)

plt.tight_layout()
plt.show()

print("Key insight: PPO takes the PESSIMISTIC (lower) bound.")
print("- For good actions: caps the benefit, preventing overly greedy updates")
print("- For bad actions: caps the penalty, preventing catastrophic policy changes")
print("- Result: a trust region without second-order optimization (unlike TRPO)")

In [ ]:
class PPO:
    """Proximal Policy Optimization with Clipped Surrogate Objective.
    
    Full implementation with:
    - Clipped surrogate objective for policy
    - GAE for advantage estimation
    - Value function trained with plain MSE (value clipping not implemented here; see notebook 05)
    - Multiple epochs of full-batch updates per rollout (mini-batching not implemented here; see notebook 05)
    """
    
    def __init__(
        self,
        obs_dim,
        act_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        ppo_epochs=4,
        value_coef=0.5,
        entropy_coef=0.01,
    ):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.ppo_epochs = ppo_epochs
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        
        self.policy = PolicyNetwork(obs_dim, act_dim).to(device)
        self.value_fn = ValueNetwork(obs_dim).to(device)
        
        # Shared optimizer for simplicity (separate optimizers also common)
        self.optimizer = torch.optim.Adam(
            list(self.policy.parameters()) + list(self.value_fn.parameters()),
            lr=lr,
        )
    
    def compute_gae(self, rewards, values, dones):
        """Compute GAE advantages and returns."""
        advantages = np.zeros_like(rewards, dtype=np.float32)
        gae = 0
        
        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0.0  # terminal
            else:
                next_value = values[t + 1]
            
            if dones[t]:
                next_value = 0.0
                gae = 0.0
            
            delta = rewards[t] + self.gamma * next_value - values[t]
            gae = delta + self.gamma * self.gae_lambda * gae
            advantages[t] = gae
        
        returns = advantages + values
        return advantages, returns
    
    def collect_episode(self, env):
        """Collect one episode of experience."""
        states, actions, rewards, log_probs, dones = [], [], [], [], []
        
        obs, _ = env.reset()
        done = False
        
        while not done:
            states.append(obs.copy())
            
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                dist = self.policy(obs_t)
                action = dist.sample()
                log_prob = dist.log_prob(action)
            
            obs, reward, terminated, truncated, _ = env.step(action.item())
            # NOTE: treating truncation like termination zeroes the bootstrap at
            # time-limit cutoffs, which biases value targets; correct handling
            # bootstraps V(s_next) when truncated (zero only on true termination).
            done = terminated or truncated
            
            actions.append(action.item())
            rewards.append(reward)
            log_probs.append(log_prob.item())
            dones.append(done)
        
        return states, actions, rewards, log_probs, dones
    
    def update(self, log_probs_unused, rewards_unused, states=None,
               actions=None, dones=None, old_log_probs=None):
        """PPO update with multiple epochs of clipped surrogate optimization."""
        # Convert to tensors
        states_t = torch.FloatTensor(np.array(states)).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        old_log_probs_t = torch.FloatTensor(old_log_probs).to(device)
        
        # Compute values and GAE
        with torch.no_grad():
            values = self.value_fn(states_t).cpu().numpy()
        
        advantages, returns = self.compute_gae(
            np.array(rewards_unused, dtype=np.float32),
            values,
            np.array(dones, dtype=np.float32),
        )
        
        advantages_t = torch.FloatTensor(advantages).to(device)
        returns_t = torch.FloatTensor(returns).to(device)
        
        # Normalize advantages
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        
        # PPO update: multiple epochs over the same data
        total_policy_loss = 0
        total_value_loss = 0
        total_entropy = 0
        
        for epoch in range(self.ppo_epochs):
            # Recompute log probs and values under current policy
            dist = self.policy(states_t)
            new_log_probs = dist.log_prob(actions_t)
            entropy = dist.entropy().mean()
            new_values = self.value_fn(states_t)
            
            # Importance sampling ratio
            ratio = torch.exp(new_log_probs - old_log_probs_t)
            
            # Clipped surrogate objective
            surr1 = ratio * advantages_t
            surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages_t
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss
            value_loss = F.mse_loss(new_values, returns_t)
            
            # Total loss = policy + value - entropy bonus
            loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy
            
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.policy.parameters()) + list(self.value_fn.parameters()),
                0.5,
            )
            self.optimizer.step()
            
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            total_entropy += entropy.item()
        
        return total_policy_loss / self.ppo_epochs


print("PPO implementation complete.")
print(f"Key hyperparameters:")
print(f"  clip_epsilon: 0.2 (standard)")
print(f"  ppo_epochs: 4 (updates per rollout)")
print(f"  gae_lambda: 0.95 (advantage estimation)")
print(f"  entropy_coef: 0.01 (exploration bonus)")

---
## 8. PPO on CartPole -- Full Comparison

Now let's train PPO and compare all methods.

In [ ]:
def train_ppo(agent, env_name='CartPole-v1', num_episodes=1000, print_every=100):
    """Train PPO agent."""
    env = gym.make(env_name)
    episode_rewards = []
    running_reward = 0
    
    for ep in range(num_episodes):
        states, actions, rewards, log_probs, dones = agent.collect_episode(env)
        loss = agent.update(
            log_probs_unused=log_probs,
            rewards_unused=rewards,
            states=states,
            actions=actions,
            dones=dones,
            old_log_probs=log_probs,
        )
        
        ep_reward = sum(rewards)
        episode_rewards.append(ep_reward)
        running_reward = 0.95 * running_reward + 0.05 * ep_reward
        
        if (ep + 1) % print_every == 0:
            print(f"Episode {ep+1:4d} | Reward: {ep_reward:6.1f} | Running: {running_reward:6.1f}")
        
        if running_reward >= 475:
            print(f"\nSolved at episode {ep+1}! Running reward: {running_reward:.1f}")
            break
    
    env.close()
    return episode_rewards


# Train PPO
print("=" * 60)
print("Training PPO on CartPole-v1")
print("=" * 60)

ppo_agent = PPO(
    obs_dim=4, act_dim=2,
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    clip_epsilon=0.2, ppo_epochs=4,
    value_coef=0.5, entropy_coef=0.01,
)
ppo_rewards = train_ppo(ppo_agent, num_episodes=1000, print_every=100)

In [ ]:
# Grand comparison: all methods
plot_rewards(
    {
        'REINFORCE': reinforce_rewards,
        'REINFORCE + Baseline': baseline_rewards,
        'Actor-Critic (GAE)': ac_rewards,
        'PPO': ppo_rewards,
    },
    'Policy Gradient Methods Comparison on CartPole-v1'
)

In [ ]:
# Quantitative comparison

def compute_stats(rewards, name):
    """Compute summary statistics for training curves."""
    last_100 = rewards[-100:] if len(rewards) >= 100 else rewards
    solved_ep = None
    running = 0
    for i, r in enumerate(rewards):
        running = 0.95 * running + 0.05 * r
        if running >= 475 and solved_ep is None:
            solved_ep = i + 1
    
    print(f"\n{name}:")
    print(f"  Total episodes: {len(rewards)}")
    print(f"  Solved at episode: {solved_ep if solved_ep else 'Not solved'}")
    print(f"  Last 100 mean: {np.mean(last_100):.1f}")
    print(f"  Last 100 std: {np.std(last_100):.1f}")
    print(f"  Last 100 min: {np.min(last_100):.0f}")
    print(f"  Last 100 max: {np.max(last_100):.0f}")


print("=" * 60)
print("QUANTITATIVE COMPARISON")
print("=" * 60)

compute_stats(reinforce_rewards, "REINFORCE")
compute_stats(baseline_rewards, "REINFORCE + Baseline")
compute_stats(ac_rewards, "Actor-Critic (GAE)")
compute_stats(ppo_rewards, "PPO")

---
## 9. "Why Does This Work?" -- Deep Questions

### Q: Why clipping instead of a KL constraint (TRPO)?

**TRPO** (Schulman et al., 2015) solves a constrained optimization problem:

$$\max_\theta \; L^{\text{IS}}(\theta) \quad \text{s.t.} \quad D_{KL}(\pi_{\text{old}} \| \pi_\theta) \leq \delta$$

This requires computing the **natural gradient** via conjugate gradient + line search -- expensive and complex.

**PPO's insight**: Clipping achieves a similar effect to a trust region, but:
- No second-order optimization needed (just first-order gradients)
- Much simpler to implement
- Works with standard deep learning infrastructure (Adam, SGD, etc.)
- Empirically performs comparably to TRPO

The clip acts as a **soft trust region**: it prevents the ratio $r_t(\theta)$ from deviating too far from 1, which implicitly constrains how much the policy can change.

### Q: Why not just use a bigger learning rate?

A large learning rate without clipping causes **policy collapse**:
1. Large gradient step makes policy very confident about certain actions
2. Next rollout is collected under this overconfident policy
3. The rollout data is biased -- only sees outcomes of the confident actions
4. The gradient estimate is biased -- reinforces the overconfident policy
5. The policy converges to a **deterministic suboptimal** policy and can never explore out

PPO's clipping prevents step 1 by limiting how much any single update can change action probabilities.

### Q: Connection to TRPO

| Feature | TRPO | PPO |
|---------|------|-----|
| Trust region | Hard KL constraint | Soft clipping |
| Optimization | Conjugate gradient + line search | Standard SGD/Adam |
| Implementation | Complex (~100s of lines) | Simple (~20 lines of core logic) |
| Performance | Strong baseline | Comparable or better |
| Scalability | Limited by CG | Scales to large models |

PPO won the practical adoption war because it's simple, effective, and compatible with standard ML tooling. This is why OpenAI used PPO (not TRPO) for RLHF in ChatGPT/InstructGPT.

### Q: Why does PPO work for LLM alignment?

In RLHF, PPO is applied to language models where:
- **State** = prompt + tokens generated so far
- **Action** = next token from vocabulary
- **Reward** = reward model score (given at end of generation)
- **KL penalty** = $\beta \cdot D_{KL}(\pi_{\theta} \| \pi_{\text{ref}})$ prevents drift from the SFT model

The clipping mechanism is especially valuable here because:
- The action space is huge (vocabulary size ~50K)
- Small probability changes in token distributions can have large effects on outputs
- Stability is crucial -- a collapsed language model produces gibberish

**Insider Tip:** PPO is being replaced by simpler methods (DPO, GRPO, KTO) for many use cases. However, understanding PPO deeply is still expected in interviews because (a) it's foundational, (b) it's still used for reasoning model training (e.g., DeepSeek-R1 uses GRPO which is a PPO variant), and (c) it tests your RL fundamentals. Recent papers to know: 'Direct Preference Optimization' (Rafailov et al. 2023), 'DeepSeekMath: GRPO' (Shao et al. 2024), and 'KTO: Model Alignment as Prospect-Theoretic Optimization' (Ethayarajh et al. 2024). In interviews, being able to articulate *when* PPO is still preferred over DPO (e.g., when you need online exploration, reasoning tasks, or process reward models) shows real depth.

---
## Interview Question Bank: Policy Gradients & PPO

*PPO is the workhorse algorithm behind RLHF. These questions test whether you understand it deeply enough to debug it in production, not just implement it from a tutorial.*

---

### Question 1: "Walk me through the PPO objective -- what is each term doing?"

**What we're testing:** Deep algorithmic understanding. Can you explain WHY each component exists, not just WHAT it is?

**Good answer:** The PPO-Clip objective is:

$L^{CLIP}(\theta) = \mathbb{E}[\min(r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t)]$

Where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio. The clipping ($\epsilon = 0.2$ typically) prevents the policy from changing too much in a single update, which stabilizes training. The advantage $\hat{A}_t$ tells us whether this action was better or worse than expected.

**Great answer (Principal-level):** All of the above, plus: explains WHY clipping is preferred over the KL penalty approach (TRPO). TRPO requires computing/approximating the Fisher information matrix and solving a constrained optimization problem -- computationally expensive and harder to implement. PPO's clipping achieves a similar trust-region effect with a simple first-order method. The key insight is that clipping creates a pessimistic lower bound: when the advantage is positive, we clip the ratio from above (don't exploit good actions too aggressively); when the advantage is negative, we clip from below (don't overcorrect bad actions). This asymmetric behavior prevents both overconfidence and overcorrection. Also discusses when PPO fails: high-dimensional action spaces, sparse rewards, and environments where the value function is hard to learn.

**Red flag:** Can't write the objective. Confuses clipping with gradient clipping. Doesn't know what the advantage function is.

**Follow-up 1:** "What's the difference between PPO-Clip and PPO-Penalty?" (PPO-Penalty replaces clipping with an adaptive KL penalty: $L = \mathbb{E}[r_t \hat{A}_t - \beta \text{KL}[\pi_{\theta_{old}} || \pi_\theta]]$, where $\beta$ is adjusted based on whether the actual KL exceeds a target. In practice, PPO-Clip is more popular because it's simpler and works well across many settings.)

**Follow-up 2:** "If I remove the clipping, what happens?" (The policy can change dramatically in a single update. This leads to policy collapse: the model finds an exploitable direction and races toward it, destroying previously learned behaviors. The training becomes unstable with oscillating performance.)

---

### Question 2: "Why use GAE instead of simple returns?"

**What we're testing:** Understanding of bias-variance trade-offs, which is fundamental to RL algorithm design.

**Good answer:** Monte Carlo returns (sum of future rewards) give unbiased estimates but have high variance because they depend on the entire trajectory. TD (Temporal Difference) errors have lower variance but are biased because they depend on the value function estimate. GAE (Generalized Advantage Estimation) interpolates between these extremes using a parameter $\lambda \in [0,1]$.

**Great answer (Principal-level):** Derives or explains GAE: $\hat{A}^{GAE(\gamma,\lambda)}_t = \sum_{l=0}^{\infty}(\gamma\lambda)^l \delta_{t+l}$ where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error. When $\lambda=1$, this equals the Monte Carlo return minus the baseline (high variance, low bias). When $\lambda=0$, this is just the one-step TD error (low variance, high bias). The connection to TD($\lambda$) is that GAE is TD($\lambda$) applied to advantage estimation. In practice, $\lambda = 0.95$ works well across many settings. For RLHF specifically, $\lambda$ is less critical because episodes are short (one generation), but GAE still helps by smoothing value estimation noise.

**Red flag:** Doesn't know what bias-variance trade-off means in this context. Can't explain what $\lambda$ controls. Confuses returns with rewards.

**Follow-up:** "How do you tune $\lambda$ in practice?" (Start with 0.95. If training is unstable, reduce it. If the policy converges too slowly to a suboptimal solution, increase it. In practice, $\lambda$ is one of the less sensitive hyperparameters compared to learning rate or clip epsilon.)

---
## Production Implementation Notes: PPO at Scale

*PPO in a research notebook vs PPO running on 1000 GPUs are very different beasts.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (frontier labs) |
|-----------|--------------------------------|-----------------------------------|
| **Environment** | CartPole (4-dim state, 2 actions) | Language generation (50K+ token vocabulary, variable-length episodes) |
| **Action space** | Discrete, 2 actions | Discrete but massive (32K-128K vocabulary at each step) |
| **Episode length** | ~200 steps | Variable, 50-2000 tokens per generation |
| **Parallelism** | Single process | 256-1024 GPUs with complex parallelism strategies |
| **Value function** | Separate small network | Additional head on the language model (shared backbone) |
| **PPO epochs** | 3-10 per batch | 1-4 (more causes overfitting on the small batch of generations) |

### Scale Numbers for PPO in RL (Pre-LLM Context)

- **OpenAI Five (Dota 2):** The 2018 setup used 256 GPUs plus 128,000 CPU cores, generating roughly 180 years of self-play experience per day; the final system trained over ~10 months. PPO with very large batches (millions of timesteps per update).
- **AlphaStar (StarCraft):** Used an IMPALA-style actor-critic with V-trace, UPGO, and TD(lambda), plus league-based self-play -- not PPO.
- **Key insight:** The scale of PPO in games dwarfs what we do in RLHF. RLHF is expensive because of the generation bottleneck, not because of PPO itself.

### Engineering Challenges Not in Papers

1. **Hyperparameter sensitivity:** PPO has ~10 hyperparameters that interact nonlinearly (learning rate, clip epsilon, GAE lambda, number of epochs, batch size, value loss coefficient, entropy bonus, max gradient norm, discount factor, minibatch size). In production, these require careful tuning per task.
2. **Value function estimation:** The value head is often poorly calibrated early in training. This makes the advantage estimates noisy, which makes the policy updates noisy. Value function warm-up (pretraining the value head before starting policy updates) helps.
3. **Advantage normalization:** Normalizing advantages to zero mean and unit variance per batch is critical for stable training. Without this, the scale of the clipping threshold is wrong.
4. **Implementation bugs:** PPO is notoriously easy to implement incorrectly. The "37 implementation details of PPO" blog post (by Costa Huang) documents subtle issues that cause 2-3x performance differences. Common bugs: wrong advantage computation order, incorrect reward normalization, not resetting hidden states.
5. **Reproducibility:** PPO training is highly sensitive to random seeds, even more so than supervised training. Results can vary by 50%+ across seeds. Production systems run multiple seeds and use robust aggregation.

### Monitoring PPO Training

- **Clip fraction:** What fraction of the probability ratios are being clipped? Should be 10-30%. If >50%, the policy is changing too fast. If <5%, learning is too slow.
- **Explained variance of value function:** How well does the critic predict returns? Should increase over training. Low explained variance means advantage estimates are noisy.
- **Entropy:** Policy entropy should decrease gradually as the policy becomes more confident. Rapid entropy collapse indicates premature convergence.
- **KL divergence:** Between old and new policy (per update). Should be small (< 0.01-0.03). Large KL means the trust region is being violated despite clipping.
- **Approximate KL:** Many implementations track this as an early stopping criterion for the PPO epoch loop.

---
## How This Gets Tested in Interviews

### Where PPO Questions Appear

| Company | Round | Format | Depth |
|---------|-------|--------|-------|
| **Anthropic** | Onsite (technical depth) | Derivation + discussion | Deep -- derive the objective, explain each design choice |
| **OpenAI** | Onsite (RL knowledge) | Whiteboard + discussion | Deep -- full PPO derivation, failure mode analysis |
| **DeepMind** | Onsite (RL fundamentals) | Mathematical derivation | Very deep -- may ask you to derive PPO from first principles starting with policy gradients |
| **Meta (GenAI)** | Onsite | Discussion + coding | Moderate -- implement PPO update, discuss trade-offs |
| **xAI / Mistral** | Technical screen | Discussion | Moderate -- practical understanding, less mathematical rigor |

### Time Expectations

- **"Derive the PPO objective from policy gradients"**: 15-20 minutes at the whiteboard. Start with REINFORCE, add baseline, explain importance sampling, arrive at clipped objective.
- **"Implement PPO update step"**: 15-20 minutes coding. Given a batch of (states, actions, old_log_probs, advantages), implement the clipped loss and update.
- **"Compare PPO, TRPO, and REINFORCE"**: 10-15 minute discussion. Explain the progression and why each improvement was needed.

### Senior vs. Principal Expectations

**senior ML engineer:**
- Write the PPO-Clip objective correctly
- Explain what clipping does and why it's needed
- Implement a PPO update given batched data
- Know GAE and explain the bias-variance trade-off
- Understand the progression: REINFORCE -> baseline -> actor-critic -> PPO

**principal ML engineer:**
- All of the above, plus:
- Derive PPO from first principles (policy gradient theorem -> importance sampling -> trust regions)
- Explain the connection between PPO and natural policy gradients / Fisher information matrix
- Discuss the "37 implementation details" -- know at least 5-10 common bugs
- Reason about when PPO fails and what alternatives exist (DPO, GRPO, REINFORCE++)
- Have an opinion: is PPO necessary for RLHF or are simpler methods (DPO) sufficient? (Strong candidates can argue both sides)
- Know the relationship between PPO hyperparameters and training dynamics (what happens if you double the clip epsilon?)

### Preparation Checklist

- [ ] Derive the PPO objective starting from the policy gradient theorem (practice at a whiteboard)
- [ ] Implement PPO update from scratch given batched data (time yourself -- should take <15 min)
- [ ] Be able to explain: why clipping and not KL penalty? (simplicity, first-order, robust)
- [ ] Know GAE derivation and the role of $\lambda$
- [ ] Read Costa Huang's "37 implementation details of PPO" -- know at least the top 10
- [ ] Be ready for: "PPO training is unstable on your task. What do you try?" (Have a systematic debugging procedure: check advantage normalization, reduce learning rate, reduce clip epsilon, increase batch size, check value function accuracy)

---
## 10. Flashcard Summary

Use these for spaced repetition review. Cover the answer column and recall from the question.

| # | Question | Answer |
|---|----------|--------|
| 1 | State the policy gradient theorem. | $\nabla_\theta J = \mathbb{E}_{\tau \sim \pi}[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t]$ -- gradient of expected return is the expected product of score function and return. |
| 2 | What is REINFORCE? | Monte Carlo policy gradient: collect full trajectory, compute returns $G_t$, update $\theta \leftarrow \theta + \alpha \sum_t \nabla \log \pi \cdot G_t$. |
| 3 | What is the main problem with REINFORCE? | High variance in gradient estimates because Monte Carlo returns are noisy. Requires many episodes for reliable gradients. |
| 4 | Why can we subtract a baseline without bias? | $\mathbb{E}_a[\nabla \log \pi(a|s) \cdot b(s)] = b(s) \nabla_\theta \sum_a \pi(a|s) = b(s) \cdot 0 = 0$. The gradient of a probability distribution summing to 1 is 0. |
| 5 | What is the advantage function? | $A(s,a) = Q(s,a) - V(s)$ -- how much better action $a$ is compared to the average action in state $s$. |
| 6 | What is GAE? Write the formula. | $\hat{A}_t^{GAE} = \sum_{l=0}^{T-t} (\gamma\lambda)^l \delta_{t+l}$ where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$. $\lambda$ trades off bias ($\lambda=0$, TD) vs. variance ($\lambda=1$, MC). |
| 7 | What is the importance sampling ratio in PPO? | $r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{old}}(a_t|s_t)$ -- ratio of new to old policy probabilities. |
| 8 | Write PPO's clipped objective. | $L^{CLIP} = \mathbb{E}[\min(r \cdot A, \text{clip}(r, 1-\epsilon, 1+\epsilon) \cdot A)]$ with $\epsilon=0.2$. Takes pessimistic (lower) bound. |
| 9 | How does clipping create a trust region? | Clipping the ratio to $[1-\epsilon, 1+\epsilon]$ limits how much each action's probability can change in one update, preventing catastrophic policy changes without second-order optimization. |
| 10 | Why PPO over TRPO? | PPO is simpler (first-order only, no conjugate gradient/line search), compatible with standard optimizers (Adam), empirically comparable performance. |
| 11 | What is the entropy bonus? | $+ c \cdot H(\pi(\cdot|s))$ encourages exploration by penalizing overly deterministic policies. Typical $c = 0.01$. |
| 12 | How is PPO applied in RLHF for LLMs? | State = prompt + generated tokens so far. Action = next token. Reward = RM score at end of generation. KL penalty against reference (SFT) policy prevents reward hacking and mode collapse. |

---
## 11. Paper Guide

### Primary Paper

**Proximal Policy Optimization Algorithms (Schulman et al., 2017)**
- https://arxiv.org/abs/1707.06347
- **Read for**: The clipped surrogate objective derivation (Section 3), comparison with TRPO, ablation studies
- **Key insight**: Simple clipping achieves trust-region-like behavior without second-order methods
- **Interview focus**: Be able to derive the clipped objective from the importance-sampled surrogate, explain why `min` is used (pessimistic bound), and discuss the connection to TRPO

### Supporting Papers

**1. Policy Gradient Methods for RL with Function Approximation (Sutton et al., 2000)**
- The original policy gradient theorem
- **Read for**: The foundational result that makes all policy gradient methods possible

**2. Simple Statistical Gradient-Following Algorithms for Connectionist RL (Williams, 1992)**
- The REINFORCE algorithm
- **Read for**: Historical context, original derivation of Monte Carlo policy gradient

**3. Trust Region Policy Optimization (Schulman et al., 2015)**
- https://arxiv.org/abs/1502.05477
- **Read for**: The constrained optimization view that motivated PPO, natural gradient, conjugate gradient
- **Interview focus**: Know the KL constraint formulation and why PPO simplified it

**4. High-Dimensional Continuous Control Using GAE (Schulman et al., 2016)**
- https://arxiv.org/abs/1506.02438
- **Read for**: The GAE derivation, bias-variance trade-off analysis with $\lambda$

**5. InstructGPT (Ouyang et al., 2022)**
- https://arxiv.org/abs/2203.02155
- **Read for**: How PPO is applied to LLMs in practice (Section 3.3), the KL penalty term, reward model interaction

### Recent Papers (2024-2025) -- Know for Interviews

**6. Direct Preference Optimization (Rafailov et al., 2023)**
- https://arxiv.org/abs/2305.18290
- **Read for**: The main alternative to PPO for RLHF. Shows the RM+PPO pipeline can be collapsed into a single supervised loss. Know the DPO vs PPO tradeoffs cold for interviews.

**7. GRPO: Group Relative Policy Optimization (Shao et al., 2024)**
- https://arxiv.org/abs/2402.03300 (DeepSeekMath)
- **Read for**: PPO variant that removes the value model by using group-based advantage estimation. Used in DeepSeek-R1 for reasoning model training.

**8. KTO: Model Alignment as Prospect-Theoretic Optimization (Ethayarajh et al., 2024)**
- https://arxiv.org/abs/2402.01306
- **Read for**: Alignment from binary (good/bad) feedback without paired preferences -- simpler data requirements than DPO

### Suggested Reading Order
1. Schulman et al. 2017 (PPO) -- the core method
2. Schulman et al. 2016 (GAE) -- the advantage estimation trick
3. Schulman et al. 2015 (TRPO) -- understand what PPO simplified
4. Ouyang et al. 2022 (InstructGPT) -- PPO for LLM alignment
5. Rafailov et al. 2023 (DPO) -- the modern alternative
6. Shao et al. 2024 (GRPO) -- the latest PPO evolution for reasoning

---
*Notebook 04 complete. You now have the complete policy gradient toolkit: REINFORCE -> baselines -> actor-critic -> GAE -> PPO.*